# Native Pythia optimizer-state reconstruction

Coupling-Phase Spectroscopy — governed Colab runner.

Downloads the raw GPT-NeoX checkpoint, reconstructs ZeRO-partitioned Adam moments, and runs CPS with the native state. This notebook is storage-heavy. Set `CPS_NATIVE_REVISION` and ensure the selected native repository exposes that branch.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import os, pathlib
from cps.pythia.checkpoints import download_native_checkpoint
revision=os.environ.get("CPS_NATIVE_REVISION", "step143000")
target=pathlib.Path(f"/content/native-pythia-70m-{revision}")
result=download_native_checkpoint("EleutherAI/neox-ckpt-pythia-70m", revision, target)
print(result)

In [ ]:
import dataclasses
from cps.pythia.config import load_probe_config
from cps.pythia.runner import run_probe
base=load_probe_config("subjects/pythia/configs/pythia_70m_native.yaml")
state=dataclasses.replace(base.state, native_checkpoint_dir=str(target), step=int(revision.removeprefix("step")))
model=dataclasses.replace(base.model, revision=revision)
output=run_probe(dataclasses.replace(base, state=state, model=model))
print(output)

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)